# Beyond the stopwatch
## Does Combine speed translate to rookie NFL routes?
**NFL Big Data Bowl 2027 · receiver study · version 2**

**The football question.** Does sensor-measured speed during the 40-yard dash
improve our understanding of a receiver's rookie route movement beyond the
official 40 time and body measurements?

This notebook follows one narrow, testable question from raw tracking to a later
draft-class evaluation. It produces four figures, audit tables, model comparisons,
and a computed research summary. Findings are generated from your attached data.

**Run in Kaggle:** upload this notebook, attach **NFL Big Data Bowl 2027** with
**Add Input**, select **CPU**, then **Run All**. Internet access and external data
are unnecessary. The large game files are scanned once; a validated local cache
speeds up reruns in the same session. Outputs appear in
`/kaggle/working/receiver_speed`.

**Run locally:** install `numpy pandas scipy scikit-learn matplotlib ipython`,
set `DATA_DIR_OVERRIDE` below to the extracted competition folder, and run all
cells. Use pandas 2 or later and scikit-learn 1.2 or later.

Competition references: [overview and submission requirements](https://www.kaggle.com/competitions/nfl-big-data-bowl-2027/overview),
[data dictionary](https://www.kaggle.com/competitions/nfl-big-data-bowl-2027/data).
Data attribution: The National Football League / Kaggle, NFL Big Data Bowl 2027.
Review the license and rules on the official pages before publishing an entry.

### Analysis contract

| Choice | Rule fixed before looking at results |
|---|---|
| Population | WRs in draft classes 2023–2025; one observation per player |
| Exposure | Draft-year regular season; non-nullified, labeled routes |
| Eligibility | Official 40, a valid tracked 40, ≥50 rookie routes, ≥20 tracked routes |
| Combine signal | Median across attempts of the largest three-frame mean speed |
| Primary outcome | Mean across routes of the largest three-frame mean speed between snap and release |
| Context sensitivity | Route speed after accounting for route label and time to release, fitted on training players |
| Exploratory outcomes | Separation at release; receiving yards per labeled route |
| Comparison | Training mean → official 40 only → official measurements → measurements plus tracking |
| Final holdout | Train on 2023–2024; evaluate on 2025 |
| Uncertainty | Paired player bootstrap of holdout errors; fitted models held fixed |

A three-frame mean reduces the influence of a single speed spike. All three
frames must belong to the same route or drill attempt and be close in time.
We do not reconstruct a stopwatch time from clip boundaries.

The primary outcome measures movement during the available route window. It
does not measure overall receiving ability. Route duration and route assignment
can change that window, which motivates the separate context sensitivity.
Career totals and draft pick are excluded from predictors. The thresholds and
ridge penalty are configurable below; avoid changing them to obtain a preferred
holdout result.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sklearn
from IPython.display import Markdown, display
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PIPELINE_VERSION = '2.0.0'
SEED = 2027
DATA_DIR_OVERRIDE = None  # Example: Path('/path/to/extracted/competition')
OUTPUT_DIR = Path('/kaggle/working/receiver_speed') if Path('/kaggle/input').exists() else Path('receiver_speed_outputs')
CACHE_DIR = Path('/kaggle/temp/receiver_speed_cache') if Path('/kaggle/input').exists() else Path('receiver_speed_work/cache')
USE_CACHE = True
EXPORT_PLAYER_TABLES = False  # Optional cohort, attempt, route, and prediction CSVs
CHUNK_SIZE = 250_000
YEARS = (2023, 2024, 2025)
TRAIN_YEARS = (2023, 2024)
HOLDOUT_YEAR = 2025
MIN_ROUTES = 50
MIN_TRACKED_ROUTES = 20
MIN_SEPARATION_ROUTES = 20
MIN_ATTEMPT_FRAMES = 15
MIN_ROUTE_FRAMES = 5
MAX_SPEED = 15.0  # yd/s; a generous tracking quality limit
FRAME_GAP_RANGE = (0.05, 0.20)  # seconds between adjacent samples
MAX_ATTEMPT_GAP = 0.30
RELEASE_RANGE = (0.3, 10.0)  # seconds after snap
MIN_TRAIN_PLAYERS = 10
MIN_TEST_PLAYERS = 5
RIDGE_ALPHA = 10.0
BOOTSTRAPS = 2000
BASE_FEATURES = ['forty', 'ten_yd_split', 'combine_height', 'combine_weight']
TRACK_FEATURES = ['combine_speed']
PRIMARY_TARGET = 'rookie_route_speed'
TARGET_INFO = {
    'rookie_route_speed': ('Route speed · primary', 'yd/s'),
    'route_adjusted_speed': ('Context-adjusted speed · sensitivity', 'yd/s'),
    'rookie_separation': ('Separation · exploratory', 'yd'),
    'rookie_yards_per_route': ('Yards per route · exploratory', 'yd/route'),
}
PLAY_KEYS = ['game_id', 'play_id']
ROUTE_KEYS = PLAY_KEYS + ['nfl_id']
ATTEMPT_KEYS = ['nfl_id', 'draft_year', 'event_id', 'attempt']
PALETTE = {2023: '#0072B2', 2024: '#D55E00', 2025: '#009E73'}
MODEL_LABELS = {'mean': 'Training mean', 'forty_only': 'Official 40',
                'official': 'Official measurements', 'with_tracking': 'Plus tracking'}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 180, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'axes.grid': True,
                     'grid.alpha': 0.15})
START = time.time()
quality_rows = []

def record(stage, check, value, detail=''):
    quality_rows.append({'stage': stage, 'check': check, 'value': int(value), 'detail': detail})

def save_figure(fig, stem):
    fig.savefig(OUTPUT_DIR / f'{stem}.png', bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / f'{stem}.svg', bbox_inches='tight')
    plt.show()

print(f'Pipeline {PIPELINE_VERSION} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}')

### 1. Inspect the files before analysis

Header checks run before scanning the large game files. Required identifiers must
be present and unique where a table promises one row per entity. Exact repeated
metadata rows are collapsed; conflicting duplicates stop the analysis.

Drill labels are checked using the columns and values actually present. Some
schemas expose `drill_type`, others also expose `drill_name`; the extraction loads
both when available. A missing field is never accessed through a DataFrame
attribute. Optional baseline measurements are reported if absent.

In [ ]:
COLUMN_ALIASES = {
    'nfl_id': ('nfl_id', 'nflId'), 'game_id': ('game_id', 'gameId'),
    'play_id': ('play_id', 'playId'), 'display_name': ('display_name', 'displayName'),
    'ten_yd_split': ('ten_yd_split', 'ten_yard_split'),
}

def normalized_label(values):
    return (values.astype('string').str.strip().str.upper()
            .str.replace(r'[^A-Z0-9]+', '_', regex=True).str.strip('_'))

def numeric_ids(frame):
    for column in ('nfl_id', 'game_id', 'play_id', 'draft_year', 'season'):
        if column in frame:
            values = pd.to_numeric(frame[column], errors='coerce')
            if (values.dropna() % 1 != 0).any():
                raise ValueError(f'{column} contains a fractional identifier.')
            frame[column] = values.astype('Int64')
    return frame

if DATA_DIR_OVERRIDE is not None:
    DATA_DIR = Path(DATA_DIR_OVERRIDE).expanduser().resolve()
else:
    search_root = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('data')
    candidates = sorted({path.parent.resolve() for path in search_root.rglob('combine_tracking.csv')})
    if len(candidates) != 1:
        raise FileNotFoundError('Attach the NFL Big Data Bowl 2027 data, or set DATA_DIR_OVERRIDE. '
                                f'Found {len(candidates)} candidate folders: {candidates}')
    DATA_DIR = candidates[0]

required_files = ['players.csv', 'combine_results.csv', 'combine_tracking.csv',
                  'player_play.csv', 'games.csv'] + [f'game_tracking_{year}.csv' for year in YEARS]
missing_files = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing competition files: {missing_files}')

headers = {name: list(pd.read_csv(DATA_DIR / name, nrows=0).columns) for name in required_files}

def column_map(name, required, optional=()):
    available = headers[name]
    mapping = {}
    for canonical in list(required) + list(optional):
        matches = [col for col in COLUMN_ALIASES.get(canonical, (canonical,)) if col in available]
        if matches:
            mapping[matches[0]] = canonical
        elif canonical in required:
            raise ValueError(f'{name}: required column {canonical!r} is absent. Available: {available}')
    return mapping

def read_table(name, required, optional=(), chunksize=None):
    mapping = column_map(name, required, optional)
    loaded = pd.read_csv(DATA_DIR / name, usecols=list(mapping), low_memory=False, chunksize=chunksize)
    if chunksize is None:
        return numeric_ids(loaded.rename(columns=mapping))
    return (numeric_ids(part.rename(columns=mapping)) for part in loaded)

def unique_metadata(frame, keys, label):
    before = len(frame)
    frame = frame.drop_duplicates().copy()
    record('metadata', f'{label}: exact duplicate rows removed', before - len(frame))
    if frame[keys].isna().any().any():
        raise ValueError(f'{label}: null keys in {keys}. Resolve identifiers before analysis.')
    if frame.duplicated(keys).any():
        raise ValueError(f'{label}: conflicting duplicate keys {keys}. Resolve before joining.')
    return frame

pp_columns = ROUTE_KEYS + ['route_ran', 'target', 'rec_yards', 'pass_result',
                         'separation_at_pass_forward', 'play_nullified_by_penalty']
column_map('player_play.csv', pp_columns)
column_map('combine_tracking.csv', ATTEMPT_KEYS + ['entity_type', 'time', 's'],
           ['drill_type', 'drill_name'])
if not {'drill_type', 'drill_name'}.intersection(headers['combine_tracking.csv']):
    raise ValueError('combine_tracking.csv has neither drill_type nor drill_name.')
for year in YEARS:
    column_map(f'game_tracking_{year}.csv', ROUTE_KEYS + ['time', 's', 'event'])

players = unique_metadata(read_table('players.csv', ['nfl_id', 'draft_year', 'nfl_position'],
                                      ['display_name']), ['nfl_id'], 'players')
if 'display_name' not in players:
    players['display_name'] = players['nfl_id'].astype(str)
results = unique_metadata(read_table('combine_results.csv', ['nfl_id', 'draft_year', 'forty'],
                                     BASE_FEATURES[1:]), ['nfl_id', 'draft_year'], 'combine_results')
games = unique_metadata(read_table('games.csv', ['game_id', 'season', 'season_type']), ['game_id'], 'games')
for feature in BASE_FEATURES:
    if feature not in results:
        results[feature] = np.nan
        print(f'Optional baseline measurement unavailable: {feature}')
    results[feature] = pd.to_numeric(results[feature], errors='coerce')
invalid_forty = results['forty'].notna() & ~results['forty'].between(3, 7)
record('metadata', 'official 40 times outside 3–7 seconds', invalid_forty.sum())
results.loc[invalid_forty, 'forty'] = np.nan
wr = players.loc[normalized_label(players['nfl_position']).eq('WR') & players['draft_year'].isin(YEARS)].copy()
if wr.empty:
    raise ValueError('No WRs in the configured draft classes. Inspect players.csv and YEARS.')
games['season_type'] = normalized_label(games['season_type'])

inventory = pd.DataFrame([{'file': name, 'size_MB': round((DATA_DIR / name).stat().st_size / 1e6, 2),
                          'columns': ', '.join(headers[name])} for name in required_files])
inventory.to_csv(OUTPUT_DIR / 'file_inventory.csv', index=False)
print('Data directory:', DATA_DIR)
display(inventory[['file', 'size_MB']])
display(wr.groupby('draft_year').size().rename('WR roster').to_frame())

### 2. Extract a speed signal with frame-level checks

Use player records from a 40-yard dash only. Conflicting speeds for the same
timestamp cause the entire affected attempt to be excluded. Identical repeated
frames are collapsed. Attempts require at least 15 usable frames, a positive
duration, and no gap larger than 0.3 seconds.

The tracked feature is a rolling speed maximum, summarized by the median across
attempts. Raw maximum speed is retained for diagnostics. Neither metric assumes
that the recorded clip begins at the official timing gate.

In [ ]:
def clean_duplicate_frames(frame, keys, stage):
    frame_keys = keys + ['time']
    repeated = frame.loc[frame.duplicated(frame_keys, keep=False)]
    bad_units = pd.DataFrame(columns=keys)
    if not repeated.empty:
        conflicts = repeated.groupby(frame_keys, dropna=False)['s'].nunique()
        bad_units = conflicts.loc[conflicts.gt(1)].reset_index()[keys].drop_duplicates()
    record(stage, 'units excluded for conflicting duplicate speeds', len(bad_units))
    if not bad_units.empty:
        is_bad = pd.MultiIndex.from_frame(frame[keys]).isin(pd.MultiIndex.from_frame(bad_units))
        frame = frame.loc[~is_bad].copy()
    before = len(frame)
    frame = frame.drop_duplicates(frame_keys).copy()
    record(stage, 'identical repeated timestamps removed', before - len(frame))
    return frame

def add_rolling_speed(frame, keys):
    frame = frame.sort_values(keys + ['time']).reset_index(drop=True)
    same_unit = frame[keys].eq(frame[keys].shift(2)).all(axis=1)
    gaps = frame['time'].diff().dt.total_seconds()
    continuous = (same_unit & gaps.between(*FRAME_GAP_RANGE)
                  & gaps.shift(1).between(*FRAME_GAP_RANGE))
    frame['rolling_speed'] = frame['s'].rolling(3, min_periods=3).mean().where(continuous)
    return frame

ct = read_table('combine_tracking.csv', ATTEMPT_KEYS + ['entity_type', 'time', 's'],
                ['drill_type', 'drill_name'])
ct = ct.loc[normalized_label(ct['entity_type']).eq('PLAYER') & ct['nfl_id'].isin(wr['nfl_id'])].copy()
forty_labels = {'FORTY_YARD_DASH', '40_YARD_DASH', '40_YD_DASH', 'FORTY_YARD', '40_YARD'}
forty_mask = pd.Series(False, index=ct.index)
matched_drill_columns = []
drill_inventory = []
for column in ('drill_type', 'drill_name'):
    if column in ct:
        labels = normalized_label(ct[column])
        drill_inventory.extend({'column': column, 'value': label, 'WR_frames': count}
                               for label, count in labels.value_counts().items())
        matches = labels.isin(forty_labels)
        if matches.any():
            matched_drill_columns.append(column)
            forty_mask |= matches
pd.DataFrame(drill_inventory).to_csv(OUTPUT_DIR / 'drill_label_inventory.csv', index=False)
if not forty_mask.any():
    display(pd.DataFrame(drill_inventory))
    raise ValueError('No recognized WR 40-yard dash labels. Inspect drill_label_inventory.csv and the data dictionary.')
print('40-yard dash label matched in:', ', '.join(matched_drill_columns))
ct = ct.loc[forty_mask].copy()
record('combine', 'WR 40 frames before quality filters', len(ct))
ct['time'] = pd.to_datetime(ct['time'], errors='coerce', utc=True, format='mixed')
ct['s'] = pd.to_numeric(ct['s'], errors='coerce')
valid = ct['time'].notna() & ct['s'].between(0, MAX_SPEED) & ct[ATTEMPT_KEYS].notna().all(axis=1)
record('combine', 'invalid speed, time, or attempt key frames excluded', (~valid).sum())
ct = clean_duplicate_frames(ct.loc[valid].copy(), ATTEMPT_KEYS, 'combine')
ct = add_rolling_speed(ct, ATTEMPT_KEYS)
ct['gap_s'] = ct.groupby(ATTEMPT_KEYS, sort=False)['time'].diff().dt.total_seconds()
attempts = ct.groupby(ATTEMPT_KEYS).agg(
    frames=('s', 'size'), robust_peak=('rolling_speed', 'max'), raw_peak=('s', 'max'),
    start=('time', 'min'), end=('time', 'max'), max_gap=('gap_s', 'max')).reset_index()
attempts['duration_s'] = (attempts['end'] - attempts['start']).dt.total_seconds()
attempts['usable'] = (attempts['frames'].ge(MIN_ATTEMPT_FRAMES) & attempts['duration_s'].gt(0)
                       & attempts['robust_peak'].gt(0) & attempts['max_gap'].le(MAX_ATTEMPT_GAP))
record('combine', 'attempts rejected by quality rules', (~attempts['usable']).sum())
valid_attempts = attempts.loc[attempts['usable']].copy()
combine_features = valid_attempts.groupby(['nfl_id', 'draft_year']).agg(
    combine_speed=('robust_peak', 'median'), combine_raw_peak=('raw_peak', 'median'),
    n_combine_attempts=('robust_peak', 'size')).reset_index()
if combine_features.empty:
    raise ValueError('No valid WR 40 attempts after quality filters. Review quality_audit and timestamps.')
print(f'{len(valid_attempts):,} usable attempts from {len(combine_features):,} WRs.')
display(valid_attempts[['frames', 'duration_s', 'robust_peak', 'raw_peak']].describe().round(3))
if EXPORT_PLAYER_TABLES:
    attempts.to_csv(OUTPUT_DIR / 'combine_attempts.csv', index=False)

In [ ]:
combine_plot = combine_features.merge(results, on=['nfl_id', 'draft_year'], validate='one_to_one')
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
markers = {2023: 'o', 2024: '^', 2025: 's'}
for year in YEARS:
    part = combine_plot.loc[combine_plot['draft_year'].eq(year)]
    axes[0].scatter(part['forty'], part['combine_speed'], color=PALETTE[year],
                    marker=markers[year], label=str(year), alpha=.8, edgecolor='white', linewidth=.4)
axes[0].set(xlabel='Official 40-yard time (s)', ylabel='Tracked speed (yd/s)',
            title='A second view of straight-line speed')
axes[0].legend(title='Draft class', frameon=False)

# Deterministic example: the smallest player ID with a valid attempt.
example = valid_attempts.sort_values(ATTEMPT_KEYS).iloc[0]
example_mask = pd.Series(True, index=ct.index)
for key in ATTEMPT_KEYS:
    example_mask &= ct[key].eq(example[key])
trace = ct.loc[example_mask].copy()
trace['elapsed_s'] = (trace['time'] - trace['time'].min()).dt.total_seconds()
example_name = wr.set_index('nfl_id').loc[example['nfl_id'], 'display_name']
axes[1].plot(trace['elapsed_s'], trace['s'], color='#9AA3AD', alpha=.8, label='Recorded speed')
axes[1].plot(trace['elapsed_s'], trace['rolling_speed'], color='#0072B2', lw=2, label='Three-frame mean')
axes[1].axhline(example['robust_peak'], color='#D55E00', ls='--', lw=1)
axes[1].set(xlabel='Seconds from first recorded frame', ylabel='Speed (yd/s)',
            title=f'Frame check: {example_name}')
axes[1].legend(frameon=False)
save_figure(fig, '01_combine_signal')
del ct, trace

### 3. Give every draft class the same rookie window

Join to `games.csv` for season and game type. Retain only the player's draft-year
regular season and plays explicitly marked as non-nullified. Unknown flags or
missing game metadata are counted and excluded. Unlabeled routes are excluded.

Missing receiving yards become zero only for a known non-target or a known
incomplete/intercepted pass. Missing yards on an unresolved target remain missing;
that player's yards-per-route outcome is withheld. Separation needs at least 20
available measurements and is averaged over those measured routes.

In [ ]:
def boolean_flag(values):
    return values.astype('string').str.strip().str.upper().map({
        'TRUE': True, 'T': True, 'Y': True, 'YES': True, '1': True, '1.0': True,
        'FALSE': False, 'F': False, 'N': False, 'NO': False, '0': False, '0.0': False,
    }).astype('boolean')

pp = unique_metadata(read_table('player_play.csv', pp_columns), ROUTE_KEYS, 'player_play')
pp = pp.loc[pp['nfl_id'].isin(wr['nfl_id'])].merge(wr, on='nfl_id', validate='many_to_one')
pp = pp.merge(games, on='game_id', how='left', validate='many_to_one', indicator=True)
record('routes', 'WR player-play rows without game metadata', pp['_merge'].eq('left_only').sum())
nullified = boolean_flag(pp['play_nullified_by_penalty'])
record('routes', 'WR player-play rows with unknown nullification flag', nullified.isna().sum())
rookie = pp.loc[pp['season'].eq(pp['draft_year']) & pp['season_type'].eq('REG')
                & nullified.eq(False).fillna(False)].copy()
rookie['route_label'] = normalized_label(rookie['route_ran'])
unlabeled = rookie['route_label'].isna() | rookie['route_label'].isin(['', 'NONE', 'NAN', 'NA', 'NULL', 'N_A', 'UNKNOWN'])
routes = rookie.loc[~unlabeled].copy()
if routes.empty:
    raise ValueError('No labeled rookie regular-season WR routes. Inspect flags, seasons, and route_ran.')
routes['rec_yards'] = pd.to_numeric(routes['rec_yards'], errors='coerce')
known_non_target = boolean_flag(routes['target']).eq(False).fillna(False)
known_no_catch = normalized_label(routes['pass_result']).isin(['I', 'IN', 'INT', 'INCOMPLETE', 'INTERCEPTION'])
zero_fill = routes['rec_yards'].isna() & (known_non_target | known_no_catch)
record('routes', 'missing receiving-yard records resolved to zero', zero_fill.sum())
routes.loc[zero_fill, 'rec_yards'] = 0.0
routes['separation_at_pass_forward'] = pd.to_numeric(routes['separation_at_pass_forward'], errors='coerce')
routes.loc[routes['separation_at_pass_forward'].lt(0), 'separation_at_pass_forward'] = np.nan
production = routes.groupby('nfl_id').agg(
    n_routes=('play_id', 'size'), n_yard_records=('rec_yards', 'count'),
    receiving_yards=('rec_yards', lambda s: s.sum(min_count=1)),
    n_separation=('separation_at_pass_forward', 'count'),
    rookie_separation=('separation_at_pass_forward', 'mean')).reset_index()
production['rookie_yards_per_route'] = production['receiving_yards'] / production['n_routes']
production.loc[production['n_yard_records'].ne(production['n_routes']), 'rookie_yards_per_route'] = np.nan
production.loc[production['n_separation'].lt(MIN_SEPARATION_ROUTES), 'rookie_separation'] = np.nan
record('routes', 'eligible rookie routes before tracking match', len(routes))
display(production[['n_routes', 'n_yard_records', 'n_separation']].describe().round(1))
del pp, rookie

### 4. Read each large tracking file once

Each chunk contributes event timestamps for eligible plays and frames for eligible
receiver routes. Event tags can appear on any tracked player. Only the filtered
receiver frames for one season are held in memory; full game files are not loaded.

Frames are joined across chunks **before** duplicate checks and rolling speed
calculations. Snap and release must each have one distinct timestamp, in order,
0.3–10 seconds apart. Conflicting duplicate route frames are excluded. A valid
route needs at least five usable frames and a continuous three-frame speed window.
The cached result is reused only when source file metadata, route eligibility,
and relevant configuration match.

In [ ]:
ROUTE_MEASUREMENT_COLUMNS = ROUTE_KEYS + ['season', 'route_label', 'release_seconds', 'peak_speed', 'raw_peak', 'frames']

def process_tracking_year(year):
    filename = f'game_tracking_{year}.csv'
    stage = f'game tracking {year}'
    meta = routes.loc[routes['season'].eq(year), ROUTE_KEYS + ['route_label']].drop_duplicates()
    if meta.empty:
        return pd.DataFrame(columns=ROUTE_MEASUREMENT_COLUMNS)
    play_index = pd.MultiIndex.from_frame(meta[PLAY_KEYS].drop_duplicates())
    player_ids = set(meta['nfl_id'].dropna())
    frame_parts, event_parts = [], []
    scanned = 0
    for chunk_number, chunk in enumerate(read_table(filename, ROUTE_KEYS + ['time', 's', 'event'], chunksize=CHUNK_SIZE), 1):
        scanned += len(chunk)
        event_mask = normalized_label(chunk['event']).isin(['BALL_SNAP', 'PASS_FORWARD'])
        event_rows = chunk.loc[event_mask, PLAY_KEYS + ['time', 'event']].copy()
        if not event_rows.empty:
            in_plays = pd.MultiIndex.from_frame(event_rows[PLAY_KEYS]).isin(play_index)
            event_rows = event_rows.loc[in_plays].drop_duplicates()
            event_parts.append(event_rows)
        selected = chunk.loc[chunk['nfl_id'].isin(player_ids), ROUTE_KEYS + ['time', 's']]
        selected = selected.merge(meta[ROUTE_KEYS], on=ROUTE_KEYS, validate='many_to_one')
        if not selected.empty:
            frame_parts.append(selected)
        if chunk_number % 10 == 0:
            print(f'{year}: {scanned:,} source rows scanned', flush=True)
    record(stage, 'source rows scanned', scanned)
    if not event_parts or not frame_parts:
        record(stage, 'plays with usable snap/release boundaries', 0)
        return pd.DataFrame(columns=ROUTE_MEASUREMENT_COLUMNS)

    event_frames = pd.concat(event_parts, ignore_index=True)
    event_frames['event'] = normalized_label(event_frames['event'])
    event_frames['time'] = pd.to_datetime(event_frames['time'], utc=True, errors='coerce', format='mixed')
    event_frames = event_frames.dropna(subset=['time']).drop_duplicates(PLAY_KEYS + ['event', 'time'])
    if event_frames.empty:
        record(stage, 'plays with usable snap/release boundaries', 0)
        return pd.DataFrame(columns=ROUTE_MEASUREMENT_COLUMNS)
    event_summary = event_frames.groupby(PLAY_KEYS + ['event'])['time'].agg(['min', 'nunique'])
    event_times = event_summary['min'].unstack('event').reindex(columns=['BALL_SNAP', 'PASS_FORWARD'])
    event_counts = event_summary['nunique'].unstack('event').reindex(columns=['BALL_SNAP', 'PASS_FORWARD'])
    bounds = event_times.rename(columns={'BALL_SNAP': 'snap_time', 'PASS_FORWARD': 'release_time'})
    for time_column in ('snap_time', 'release_time'):
        bounds[time_column] = pd.to_datetime(bounds[time_column], errors='coerce', utc=True)
    bounds['release_seconds'] = (bounds['release_time'] - bounds['snap_time']).dt.total_seconds()
    valid_bounds = (event_counts.eq(1).all(axis=1) & bounds['release_seconds'].between(*RELEASE_RANGE))
    record(stage, 'plays with ambiguous or missing/out-of-range event boundaries', len(play_index) - valid_bounds.sum())
    bounds = bounds.loc[valid_bounds].reset_index()
    record(stage, 'plays with usable snap/release boundaries', len(bounds))
    if bounds.empty:
        return pd.DataFrame(columns=ROUTE_MEASUREMENT_COLUMNS)

    frames = pd.concat(frame_parts, ignore_index=True)
    del frame_parts, event_parts, event_frames
    frames['time'] = pd.to_datetime(frames['time'], utc=True, errors='coerce', format='mixed')
    frames['s'] = pd.to_numeric(frames['s'], errors='coerce')
    valid = frames['time'].notna() & frames['s'].between(0, MAX_SPEED)
    record(stage, 'invalid receiver speed/time frames excluded', (~valid).sum())
    frames = frames.loc[valid].merge(bounds, on=PLAY_KEYS, validate='many_to_one')
    frames = frames.loc[frames['time'].between(frames['snap_time'], frames['release_time'])].copy()
    frames = clean_duplicate_frames(frames, ROUTE_KEYS, stage)
    frames = add_rolling_speed(frames, ROUTE_KEYS)
    measured = frames.groupby(ROUTE_KEYS).agg(
        peak_speed=('rolling_speed', 'max'), raw_peak=('s', 'max'), frames=('s', 'size')).reset_index()
    valid_route = measured['frames'].ge(MIN_ROUTE_FRAMES) & measured['peak_speed'].notna()
    record(stage, 'routes rejected for insufficient usable frames', (~valid_route).sum())
    measured = measured.loc[valid_route].merge(meta, on=ROUTE_KEYS, validate='one_to_one')
    measured = measured.merge(bounds[PLAY_KEYS + ['release_seconds']], on=PLAY_KEYS, validate='many_to_one')
    measured['season'] = year
    record(stage, 'usable receiver routes', len(measured))
    print(f'{year}: {scanned:,} rows → {len(measured):,} measured rookie routes', flush=True)
    return measured[ROUTE_MEASUREMENT_COLUMNS]

source_signature = [{'file': name, 'bytes': (DATA_DIR / name).stat().st_size,
                     'mtime_ns': (DATA_DIR / name).stat().st_mtime_ns, 'header': headers[name]}
                    for name in required_files]
route_hash_frame = routes[ROUTE_KEYS + ['season', 'route_label']].sort_values(ROUTE_KEYS).reset_index(drop=True)
route_hash = hashlib.sha256(pd.util.hash_pandas_object(route_hash_frame, index=False).values.tobytes()).hexdigest()
cache_contract = {'version': PIPELINE_VERSION, 'sources': source_signature,
                  'route_hash': route_hash, 'minimum_frames': MIN_ROUTE_FRAMES,
                  'max_speed': MAX_SPEED, 'frame_gaps': FRAME_GAP_RANGE, 'release_range': RELEASE_RANGE}
fingerprint = hashlib.sha256(json.dumps(cache_contract, sort_keys=True).encode()).hexdigest()
cache_path = CACHE_DIR / f'routes_{fingerprint}.csv.gz'
cache_audit_path = CACHE_DIR / f'audit_{fingerprint}.json'
if USE_CACHE and cache_path.is_file() and cache_audit_path.is_file():
    route_measurements = numeric_ids(pd.read_csv(cache_path))
    quality_rows.extend(json.loads(cache_audit_path.read_text(encoding='utf-8')))
    print(f'Reused validated tracking cache: {len(route_measurements):,} routes')
else:
    audit_start = len(quality_rows)
    pieces = [process_tracking_year(year) for year in YEARS]
    nonempty = [part for part in pieces if not part.empty]
    route_measurements = (pd.concat(nonempty, ignore_index=True) if nonempty
                          else pd.DataFrame(columns=ROUTE_MEASUREMENT_COLUMNS))
    if USE_CACHE:
        route_measurements.to_csv(cache_path, index=False, compression='gzip')
        cache_audit_path.write_text(json.dumps(quality_rows[audit_start:], indent=2), encoding='utf-8')
    del pieces, nonempty
quality_audit = pd.DataFrame(quality_rows)
quality_audit.to_csv(OUTPUT_DIR / 'quality_audit.csv', index=False)
display(quality_audit.loc[quality_audit['stage'].str.startswith('game tracking')])

### 5. Show who survives the filters

Selection is part of the result. Missing Combine attempts, limited rookie playing
time, and missing event tags can change which receivers enter the study. Coverage
is the fraction of labeled routes with usable tracking after all frame checks.
No threshold is relaxed automatically when the cohort is small.

In [ ]:
if not route_measurements.empty:
    speed_outcomes = route_measurements.groupby('nfl_id').agg(
        rookie_route_speed=('peak_speed', 'mean'), n_tracked_routes=('play_id', 'size'),
        median_release_seconds=('release_seconds', 'median')).reset_index()
else:
    speed_outcomes = pd.DataFrame(columns=['nfl_id', 'rookie_route_speed', 'n_tracked_routes', 'median_release_seconds'])
cohort = (wr.merge(results, on=['nfl_id', 'draft_year'], how='left', validate='one_to_one')
          .merge(combine_features, on=['nfl_id', 'draft_year'], how='left', validate='one_to_one')
          .merge(production, on='nfl_id', how='left', validate='one_to_one')
          .merge(speed_outcomes, on='nfl_id', how='left', validate='one_to_one'))
cohort['tracking_coverage'] = cohort['n_tracked_routes'].fillna(0) / cohort['n_routes']
stages = [('WR roster', pd.Series(True, index=cohort.index)),
          ('Official 40', cohort['forty'].notna()),
          ('Usable tracked 40', cohort['combine_speed'].notna()),
          (f'≥{MIN_ROUTES} rookie routes', cohort['n_routes'].ge(MIN_ROUTES)),
          (f'≥{MIN_TRACKED_ROUTES} tracked routes', cohort['n_tracked_routes'].ge(MIN_TRACKED_ROUTES))]
flow_rows = []
included = pd.Series(True, index=cohort.index)
for stage, rule in stages:
    included &= rule.fillna(False)
    for year in YEARS:
        flow_rows.append({'draft_year': year, 'stage': stage,
                          'players': int((included & cohort['draft_year'].eq(year)).sum())})
cohort['eligible'] = included
analysis = cohort.loc[cohort['eligible']].copy()
flow = pd.DataFrame(flow_rows)
flow.to_csv(OUTPUT_DIR / 'cohort_flow.csv', index=False)
audit = cohort.groupby('draft_year').agg(
    roster=('nfl_id', 'size'), usable_tracked_40=('combine_speed', 'count'),
    usable_route_speed=('rookie_route_speed', 'count'), eligible=('eligible', 'sum'),
    median_route_coverage=('tracking_coverage', 'median'))
audit.to_csv(OUTPUT_DIR / 'cohort_audit.csv')
display(audit.round(3))
print(f'Final study cohort: {len(analysis):,} WRs')
if EXPORT_PLAYER_TABLES:
    cohort.to_csv(OUTPUT_DIR / 'analysis_cohort.csv', index=False)
    route_measurements.to_csv(OUTPUT_DIR / 'route_measurements.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
stage_names = [item[0] for item in stages]
pos = np.arange(len(stage_names))
for i, year in enumerate(YEARS):
    counts = flow.loc[flow['draft_year'].eq(year), 'players'].to_numpy()
    axes[0].barh(pos + (i - 1) * .24, counts, height=.22, color=PALETTE[year], label=str(year))
axes[0].set(yticks=pos, yticklabels=stage_names, xlabel='Receivers retained', title='Make exclusions visible')
axes[0].invert_yaxis()
axes[0].legend(frameon=False, loc='lower left', bbox_to_anchor=(0, 1.09), ncol=3)
for i, year in enumerate(YEARS):
    values = cohort.loc[cohort['draft_year'].eq(year) & cohort['n_routes'].gt(0), 'tracking_coverage'].dropna()
    if len(values):
        jitter = np.random.default_rng(SEED + year).uniform(-.12, .12, len(values))
        axes[1].scatter(np.full(len(values), i) + jitter, values, color=PALETTE[year], alpha=.65, s=22)
        axes[1].plot([i - .18, i + .18], [values.median()] * 2, color='black', lw=2)
axes[1].set(xticks=range(len(YEARS)), xticklabels=YEARS, ylim=(-.03, 1.03),
            ylabel='Usable tracked routes / labeled routes', title='Coverage among WRs with rookie routes')
save_figure(fig, '02_cohort_and_coverage')

### 6. Compare simple models on a later class

Every model uses the same players for a given outcome and split. Median
imputation, missingness indicators, and scaling are fitted on training players.
An optional measurement with no training observations is omitted and recorded.
The ridge penalty stays fixed at 10; the holdout is not used for tuning.

**Context sensitivity.** A separate ridge model estimates route speed from route
label, release duration, and squared release duration. It is fitted only on
training players' routes, with equal total weight per player. Its residuals are
averaged by player to define the context-adjusted outcome for both training and
holdout players. Holdout speeds never fit this adjustment. This checks two
observable context variables; it cannot remove every team or coverage effect.

Leave-one-class-out results are labeled sensitivity analyses. The splits holding
out 2023 or 2024 use later classes in training and are not forecasting evidence.
If a split is too small, the notebook records why it was skipped and continues.

In [ ]:
def make_player_model():
    return Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                     ('scale', StandardScaler()),
                     ('regression', Ridge(alpha=RIDGE_ALPHA, solver='lsqr'))])

context_cache = {}
def context_outcomes(train_ids, test_ids, cache_key):
    if cache_key in context_cache:
        return context_cache[cache_key]
    route_data = route_measurements.loc[route_measurements['nfl_id'].isin(set(train_ids) | set(test_ids))].copy()
    route_data['release_seconds_sq'] = route_data['release_seconds'] ** 2
    training = route_data.loc[route_data['nfl_id'].isin(train_ids)].copy()
    if training.empty:
        return pd.Series(dtype=float)
    transforms = ColumnTransformer([
        ('route', OneHotEncoder(handle_unknown='ignore'), ['route_label']),
        ('duration', StandardScaler(), ['release_seconds', 'release_seconds_sq'])], sparse_threshold=1.0)
    adjustment = Pipeline([('context', transforms), ('regression', Ridge(alpha=RIDGE_ALPHA, solver='lsqr'))])
    player_n = training.groupby('nfl_id')['nfl_id'].transform('size')
    weights = (1.0 / player_n).to_numpy()
    weights *= len(weights) / weights.sum()
    context_features = ['route_label', 'release_seconds', 'release_seconds_sq']
    adjustment.fit(training[context_features], training['peak_speed'], regression__sample_weight=weights)
    route_data['residual_speed'] = route_data['peak_speed'] - adjustment.predict(route_data[context_features])
    outcomes = route_data.groupby('nfl_id')['residual_speed'].mean()
    context_cache[cache_key] = outcomes
    return outcomes

metric_columns = ['target', 'model', 'test_class', 'train_classes', 'n_train', 'n_test', 'MAE', 'RMSE', 'R2']
status_rows, feature_rows = [], []

def evaluate_split(data, target, test_year, train_years, evaluation):
    split = data.loc[data['draft_year'].isin(tuple(train_years) + (test_year,))].copy()
    train_ids = set(split.loc[split['draft_year'].isin(train_years), 'nfl_id'])
    test_ids = set(split.loc[split['draft_year'].eq(test_year), 'nfl_id'])
    if target == 'route_adjusted_speed':
        outcomes = context_outcomes(train_ids, test_ids, (tuple(train_years), test_year))
        split[target] = split['nfl_id'].map(outcomes)
    complete = split.dropna(subset=[target])
    train = complete.loc[complete['draft_year'].isin(train_years)]
    test = complete.loc[complete['draft_year'].eq(test_year)]
    reason = 'ok' if len(train) >= MIN_TRAIN_PLAYERS and len(test) >= MIN_TEST_PLAYERS else 'insufficient players'
    status_rows.append({'evaluation': evaluation, 'target': target, 'test_class': test_year,
                        'n_train': len(train), 'n_test': len(test), 'status': reason})
    if reason != 'ok':
        return [], pd.DataFrame()
    y_train, y_test = train[target].astype(float), test[target].astype(float)
    predictions = {'mean': np.full(len(test), y_train.mean())}
    feature_sets = {'forty_only': ['forty'], 'official': BASE_FEATURES,
                    'with_tracking': BASE_FEATURES + TRACK_FEATURES}
    for model_name, requested in feature_sets.items():
        features = [name for name in requested if train[name].notna().any()]
        feature_rows.append({'evaluation': evaluation, 'target': target, 'test_class': test_year,
                             'model': model_name, 'used': ', '.join(features),
                             'unavailable_in_training': ', '.join(name for name in requested if name not in features)})
        estimator = make_player_model()
        estimator.fit(train[features].astype(float), y_train)
        predictions[model_name] = estimator.predict(test[features].astype(float))
    rows = []
    for name, prediction in predictions.items():
        rows.append({'target': target, 'model': name, 'test_class': test_year,
                     'train_classes': ','.join(map(str, train_years)), 'n_train': len(train), 'n_test': len(test),
                     'MAE': mean_absolute_error(y_test, prediction),
                     'RMSE': np.sqrt(mean_squared_error(y_test, prediction)), 'R2': r2_score(y_test, prediction)})
    out = test[['nfl_id', 'display_name', 'draft_year']].copy()
    out['target'] = target
    out['actual'] = y_test.to_numpy()
    for name, prediction in predictions.items():
        out[name] = prediction
    return rows, out

metric_rows, prediction_parts = [], []
for target in TARGET_INFO:
    rows, prediction = evaluate_split(analysis, target, HOLDOUT_YEAR, TRAIN_YEARS, 'chronological')
    metric_rows.extend(rows)
    if not prediction.empty:
        prediction_parts.append(prediction)
metrics = pd.DataFrame(metric_rows, columns=metric_columns)
holdout = (pd.concat(prediction_parts, ignore_index=True) if prediction_parts
           else pd.DataFrame(columns=['nfl_id', 'display_name', 'draft_year', 'target', 'actual'] + list(MODEL_LABELS)))
metrics.to_csv(OUTPUT_DIR / 'chronological_metrics.csv', index=False)

sensitivity_rows = []
for year in YEARS:
    for target in TARGET_INFO:
        rows, _ = evaluate_split(analysis, target, year, tuple(y for y in YEARS if y != year), 'leave_class_out')
        sensitivity_rows.extend(rows)
sensitivity = pd.DataFrame(sensitivity_rows, columns=metric_columns)
sensitivity.to_csv(OUTPUT_DIR / 'leave_class_out_metrics.csv', index=False)
model_status = pd.DataFrame(status_rows)
model_status.to_csv(OUTPUT_DIR / 'model_status.csv', index=False)
pd.DataFrame(feature_rows, columns=['evaluation', 'target', 'test_class', 'model',
                                    'used', 'unavailable_in_training']).to_csv(OUTPUT_DIR / 'model_features.csv', index=False)
if not metrics.empty:
    display(metrics.round(3))
display(model_status.loc[model_status['evaluation'].eq('chronological')])
if EXPORT_PLAYER_TABLES:
    holdout.to_csv(OUTPUT_DIR / 'holdout_predictions.csv', index=False)

In [ ]:
association_rows = []
for year in ('pooled',) + YEARS:
    part = analysis if year == 'pooled' else analysis.loc[analysis['draft_year'].eq(year)]
    paired = part[['combine_speed', PRIMARY_TARGET]].dropna()
    rho = np.nan
    if len(paired) >= 5 and paired.nunique().min() > 1:
        rho = spearmanr(paired['combine_speed'], paired[PRIMARY_TARGET]).statistic
    association_rows.append({'draft_class': year, 'n': len(paired), 'spearman_rho': rho})
associations = pd.DataFrame(association_rows)
associations.to_csv(OUTPUT_DIR / 'descriptive_associations.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for year in YEARS:
    part = analysis.loc[analysis['draft_year'].eq(year)]
    axes[0].scatter(part['combine_speed'], part[PRIMARY_TARGET], color=PALETTE[year],
                    marker=markers[year], label=f'{year} (n={len(part)})', alpha=.8)
axes[0].set(xlabel='Combine speed, median across attempts (yd/s)',
            ylabel='Mean rookie pre-release route speed (yd/s)', title='Does movement translate?')
axes[0].legend(frameon=False, title='Draft class')
p = holdout.loc[holdout['target'].eq(PRIMARY_TARGET)]
if not p.empty:
    all_values = np.r_[p['actual'], p['official'], p['with_tracking']]
    lims = (all_values.min() - .2, all_values.max() + .2)
    axes[1].plot(lims, lims, color='gray', lw=1, ls='--')
    axes[1].scatter(p['actual'], p['official'], color='#9AA3AD', marker='x', label='Official measurements')
    axes[1].scatter(p['actual'], p['with_tracking'], color=PALETTE[HOLDOUT_YEAR], label='Plus tracking')
    axes[1].set(xlim=lims, ylim=lims)
    axes[1].legend(frameon=False)
else:
    axes[1].text(.5, .5, 'Insufficient holdout players\nSee model_status.csv', ha='center', va='center', transform=axes[1].transAxes)
axes[1].set(xlabel='Observed route speed (yd/s)', ylabel='Predicted route speed (yd/s)',
            title=f'{HOLDOUT_YEAR}: receivers outside training')
save_figure(fig, '03_translation_and_holdout')

### 7. Ask how much tracking adds, with uncertainty

For each holdout player, compare the absolute errors from the official-measurement
model and the model with tracking. Positive **ΔMAE** means tracking reduced error.
Resample whole players with replacement, preserving this pairing.

These 95% percentile intervals describe sampling uncertainty in the observed
holdout with fitted models held fixed. They do not include model-training
uncertainty, data selection, or the uncertainty of a new draft class. Each panel
uses its own outcome units. Only the first outcome is primary; the others do not
provide additional independent confirmations.

In [ ]:
uncertainty_rows = []
for index, target in enumerate(TARGET_INFO):
    part = holdout.loc[holdout['target'].eq(target)]
    if part.empty:
        continue
    official_error = np.abs(part['actual'].to_numpy() - part['official'].to_numpy())
    tracking_error = np.abs(part['actual'].to_numpy() - part['with_tracking'].to_numpy())
    paired_delta = official_error - tracking_error
    rng = np.random.default_rng(SEED + index)
    draws = rng.integers(0, len(part), size=(BOOTSTRAPS, len(part)))
    bootstrap = paired_delta[draws].mean(axis=1)
    lower, upper = np.quantile(bootstrap, [.025, .975])
    uncertainty_rows.append({'target': target, 'n_holdout': len(part),
                             'official_MAE': official_error.mean(), 'tracking_MAE': tracking_error.mean(),
                             'delta_MAE': paired_delta.mean(), 'lower_95': lower, 'upper_95': upper,
                             'relative_MAE_reduction_pct': 100 * paired_delta.mean() / official_error.mean()
                             if official_error.mean() > 0 else np.nan})
uncertainty = pd.DataFrame(uncertainty_rows, columns=['target', 'n_holdout', 'official_MAE', 'tracking_MAE',
                                                   'delta_MAE', 'lower_95', 'upper_95', 'relative_MAE_reduction_pct'])
uncertainty.to_csv(OUTPUT_DIR / 'paired_error_uncertainty.csv', index=False)
display(uncertainty.round(4))

fig, axes = plt.subplots(2, 2, figsize=(11, 6), constrained_layout=True)
for ax, (target, (label, unit)) in zip(axes.flat, TARGET_INFO.items()):
    result = uncertainty.loc[uncertainty['target'].eq(target)]
    ax.axvline(0, color='gray', ls='--', lw=1)
    if not result.empty:
        row = result.iloc[0]
        color = '#0072B2' if target == PRIMARY_TARGET else '#717B86'
        ax.hlines(0, row['lower_95'], row['upper_95'], color=color, lw=3)
        ax.scatter([row['delta_MAE']], [0], color=color, s=55, zorder=3)
        ax.text(.02, .85, f"n={int(row['n_holdout'])}; ΔMAE={row['delta_MAE']:.3f}", transform=ax.transAxes)
    else:
        ax.text(.5, .55, 'Insufficient holdout players', ha='center', transform=ax.transAxes)
    ax.set(title=label, xlabel=f'Official MAE − MAE with tracking ({unit})', yticks=[], ylim=(-.7, .7))
save_figure(fig, '04_incremental_value')

### 8. Turn computed evidence into a football message

Use the primary holdout result first. A positive estimate with an interval crossing
zero is inconclusive. A negative estimate is a useful finding: this tracked feature
did not improve that baseline on the observed class. Compare the context
sensitivity before arguing that the relationship reflects the athlete rather
than route opportunity.

**Practical use:** if evidence is promising, a scouting department can flag
disagreement between stopwatch and tracked speed for film review. Any individual
player decision needs film, health, role, and sample-size context.

**Limitations to carry into the write-up:** Combine opt-outs and minimum-route
rules select the sample; only one later class is held out; route labels and release
duration leave unmeasured coverage and team effects; separation may be selectively
available; and speed translation is distinct from receiving production. These
associations do not establish a causal effect of Combine performance.

In [ ]:
primary = uncertainty.loc[uncertainty['target'].eq(PRIMARY_TARGET)]
summary_lines = [
    'BEYOND THE STOPWATCH — COMPUTED RESEARCH SUMMARY',
    f'Eligible receivers: {len(analysis)}. Training classes: {TRAIN_YEARS}; holdout: {HOLDOUT_YEAR}.',
    'Primary question: does tracked Combine speed add information beyond official measurements?',
]
if primary.empty:
    summary_lines.append('Primary comparison unavailable: the fixed minimum sample sizes were not met. '
                         'Inspect cohort_audit.csv and model_status.csv; do not infer a tracking effect.')
else:
    row = primary.iloc[0]
    verdict = ('The interval favors adding tracking in this holdout.' if row['lower_95'] > 0 else
               'The interval favors the official-measurement baseline in this holdout.' if row['upper_95'] < 0 else
               'The interval includes zero; the incremental benefit is inconclusive.')
    summary_lines.extend([
        f"Primary holdout n={int(row['n_holdout'])}; official-model MAE={row['official_MAE']:.3f} yd/s; "
        f"MAE with tracking={row['tracking_MAE']:.3f} yd/s.",
        f"Paired ΔMAE={row['delta_MAE']:.3f} yd/s, conditional 95% bootstrap interval "
        f"[{row['lower_95']:.3f}, {row['upper_95']:.3f}]. {verdict}",
    ])
for target in ('route_adjusted_speed', 'rookie_separation', 'rookie_yards_per_route'):
    part = uncertainty.loc[uncertainty['target'].eq(target)]
    if not part.empty:
        row = part.iloc[0]
        label, unit = TARGET_INFO[target]
        summary_lines.append(f"{label}: n={int(row['n_holdout'])}; ΔMAE={row['delta_MAE']:.3f} {unit}; "
                             f"interval [{row['lower_95']:.3f}, {row['upper_95']:.3f}].")
summary_lines.extend([
    'Football application: use disagreement between the stopwatch and sensor speed to prioritize film review.',
    'Interpretation: movement translation is an association, not a causal or overall talent score.',
    'Limits: selected WR cohort; one later draft class; route/team context; incomplete measurement coverage. '
    'Bootstrap intervals hold fitted models fixed.',
])
summary_text = '\n\n'.join(summary_lines)
display(Markdown('### Computed summary\n\n' + '\n\n'.join(summary_lines[1:])))
(OUTPUT_DIR / 'research_summary.txt').write_text(summary_text, encoding='utf-8')

manifest = {
    'pipeline_version': PIPELINE_VERSION, 'competition': 'nfl-big-data-bowl-2027',
    'data_directory': str(DATA_DIR), 'source_file_metadata': source_signature,
    'tracking_cache_signature': fingerprint, 'matched_drill_columns': matched_drill_columns,
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__,
    'seed': SEED, 'train_years': TRAIN_YEARS, 'holdout_year': HOLDOUT_YEAR,
    'min_routes': MIN_ROUTES, 'min_tracked_routes': MIN_TRACKED_ROUTES,
    'min_attempt_frames': MIN_ATTEMPT_FRAMES, 'min_route_frames': MIN_ROUTE_FRAMES,
    'frame_gap_range': FRAME_GAP_RANGE, 'max_attempt_gap': MAX_ATTEMPT_GAP,
    'max_speed': MAX_SPEED, 'release_range': RELEASE_RANGE,
    'min_separation_routes': MIN_SEPARATION_ROUTES,
    'min_train_players': MIN_TRAIN_PLAYERS, 'min_test_players': MIN_TEST_PLAYERS,
    'ridge_alpha': RIDGE_ALPHA, 'bootstrap_replicates': BOOTSTRAPS,
    'base_features_requested': BASE_FEATURES, 'tracking_features': TRACK_FEATURES,
    'eligible_receivers': len(analysis), 'elapsed_seconds': round(time.time() - START, 1),
}
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
pd.DataFrame(quality_rows).to_csv(OUTPUT_DIR / 'quality_audit.csv', index=False)
print(f'Completed in {time.time() - START:.1f} seconds. Outputs: {OUTPUT_DIR.resolve()}')
display(pd.DataFrame([{'output': p.name, 'size_KB': round(p.stat().st_size / 1024, 1)}
                      for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]))

### Prepare the competition write-up

1. Run this notebook on the official data and inspect `quality_audit.csv`,
   `cohort_audit.csv`, and `model_status.csv` before interpreting an effect.
2. Lead with the primary holdout comparison. Use the route-context sensitivity
   to qualify the football explanation. Describe secondary outcomes as exploratory.
3. Select figures and tables that support one clear finding. The notebook saves
   PNG and SVG versions of four figures; appendix diagnostics need not all appear
   in the write-up.
4. Edit the computed summary into your own research narrative, explaining the
   decision a coach or scout could make and the remaining limitations.
5. Follow the current official requirements: at most 2,000 words, fewer than ten
   tables/figures, and a public executed Kaggle Notebook attached to the write-up.
   Confirm the selected track and current rules on the official page.

`research_summary.txt` is a computed draft for review. Tables of model predictions
are optional analysis outputs. This analytics competition is evaluated through
its write-up; the notebook does not invent a prediction-submission format.